# Capitulo 10 - Redes neuronales recurrentes y series temporales

### Notebook: `cap10_rnns.ipynb`

Este notebook acompana al Capitulo 10 de *IA para Estudiantes de Ciencias*. Desarrolla el ejemplo central del capitulo, el **atractor de Lorenz**, y cumple las tres tareas que anuncia la caja de descripcion del capitulo:

1. **Genera trayectorias del atractor de Lorenz** mediante integracion Runge-Kutta de cuarto orden (RK4).
2. **Entrena LSTM y GRU** para predecir la evolucion del sistema y compara ambas arquitecturas.
3. **Estudia el horizonte de predictibilidad en funcion de la longitud del contexto** $L$, y lo contrasta con el limite teorico impuesto por el exponente de Lyapunov.

El sistema de Lorenz es deterministico pero caotico: dos condiciones iniciales arbitrariamente proximas divergen exponencialmente. Esto impone un horizonte de predictibilidad finito que ninguna arquitectura puede superar, y que sirve de referencia fisica para evaluar las redes.

**Compatibilidad.** El notebook funciona en Colab, JupyterLab, VS Code y Kaggle. Detecta GPU automaticamente, pero se ejecuta tambien en CPU (con tiempos mayores).


## 0. Entorno y reproducibilidad

Se fijan las semillas para que los resultados sean reproducibles y se detecta el dispositivo de calculo.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

# Reproducibilidad
SEMILLA = 0
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)

dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", dispositivo)
print("Version de PyTorch:", torch.__version__)

## 1. Generacion del atractor de Lorenz con RK4

El sistema de Lorenz es un modelo muy simplificado de la conveccion atmosferica:

$$\frac{dx}{dt} = \sigma(y - x), \qquad
  \frac{dy}{dt} = x(\rho - z) - y, \qquad
  \frac{dz}{dt} = xy - \beta z,$$

con los parametros clasicos $\sigma = 10$, $\rho = 28$, $\beta = 8/3$. En este regimen el sistema exhibe el famoso atractor en forma de mariposa.

La integracion usa Runge-Kutta de cuarto orden (RK4). Tras descartar un transitorio inicial para que la trayectoria caiga sobre el atractor, se obtiene una serie de 45 000 estados que servira de conjunto de datos.

> Este es el primer listado de codigo del capitulo (*Generacion del atractor de Lorenz con RK4*).

In [ ]:
def lorenz_derivadas(u, sigma=10., rho=28., beta=8./3.):
    """Calcula (dx/dt, dy/dt, dz/dt) para el sistema de Lorenz."""
    x, y, z = u
    return np.array([sigma*(y - x),
                     x*(rho - z) - y,
                     x*y - beta*z])

def rk4_paso(f, u, dt):
    """Un paso de Runge-Kutta de orden 4."""
    k1 = f(u)
    k2 = f(u + dt/2 * k1)
    k3 = f(u + dt/2 * k2)
    k4 = f(u + dt   * k3)
    return u + dt/6 * (k1 + 2*k2 + 2*k3 + k4)

# Generar trayectoria
dt   = 0.01
N    = 50_000
u0   = np.array([1.0, 1.0, 1.0])
traj = np.zeros((N, 3))
traj[0] = u0
for i in range(1, N):
    traj[i] = rk4_paso(lorenz_derivadas, traj[i-1], dt)

# Descartar transitorio inicial (primeros 5000 pasos)
traj = traj[5000:]

print(f"Forma de la trayectoria: {traj.shape}")  # (45000, 3)
print(f"Media: {traj.mean(axis=0).round(2)}")
print(f"Std:   {traj.std(axis=0).round(2)}")

### Visualizacion del atractor

Representamos la trayectoria en 3D y las tres componentes en funcion del tiempo. La estructura de dos lobulos (las "alas" de la mariposa) corresponde a dos puntos fijos inestables entre los que el sistema salta de forma impredecible.

In [ ]:
fig = plt.figure(figsize=(11, 4))

# Atractor 3D
ax = fig.add_subplot(1, 2, 1, projection="3d")
ax.plot(traj[:, 0], traj[:, 1], traj[:, 2], lw=0.3, color="#1E5FA5")
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
ax.set_title("Atractor de Lorenz")

# Componentes vs tiempo (primeros 3000 pasos)
ax2 = fig.add_subplot(1, 2, 2)
t = np.arange(3000) * dt
ax2.plot(t, traj[:3000, 0], lw=0.8, label="x")
ax2.plot(t, traj[:3000, 1], lw=0.8, label="y")
ax2.plot(t, traj[:3000, 2], lw=0.8, label="z")
ax2.set_xlabel("tiempo simulado")
ax2.set_ylabel("estado")
ax2.set_title("Componentes frente al tiempo")
ax2.legend()
plt.tight_layout()
plt.show()

## 2. Preprocesado: normalizacion y ventanas deslizantes

La red aprende a predecir el estado $\mathbf{u}_{t+1}$ dado el contexto de los $L$ pasos anteriores. Esto se implementa con **ventanas deslizantes**: para cada posicion $t$, la entrada es la secuencia $[\mathbf{u}_t, \ldots, \mathbf{u}_{t+L-1}]$ y el objetivo es el siguiente estado $\mathbf{u}_{t+L}$.

Dos buenas practicas del capitulo son cruciales aqui:

- **Split temporal estricto.** El conjunto de test corresponde a instantes *posteriores* en el tiempo al de entrenamiento, nunca a intervalos aleatorios intercalados.
- **Normalizacion con la estadistica del entrenamiento.** La media y la desviacion tipica se calculan solo sobre el tramo de entrenamiento y se aplican a validacion y test.

> Inicio del segundo listado del capitulo (*LSTM para prediccion del atractor de Lorenz*).

In [ ]:
# ----- Normalizacion (solo con la estadistica del entrenamiento) -----
n_train_raw = int(0.7 * len(traj))
media = traj[:n_train_raw].mean(axis=0)
std   = traj[:n_train_raw].std(axis=0)
traj_norm = (traj - media) / std

# ----- Ventanas deslizantes -----
L = 40  # longitud del contexto (pasos de tiempo)

def crear_ventanas(datos, L):
    X = np.array([datos[i:i+L]   for i in range(len(datos)-L)])
    y = np.array([datos[i+L]     for i in range(len(datos)-L)])
    return X, y

X_all, y_all = crear_ventanas(traj_norm, L)

# Split temporal (NO aleatorio!)
n_total = len(X_all)
n_train = int(0.7 * n_total)
n_val   = int(0.15 * n_total)

X_train = torch.tensor(X_all[:n_train],         dtype=torch.float32)
y_train = torch.tensor(y_all[:n_train],         dtype=torch.float32)
X_val   = torch.tensor(X_all[n_train:n_train+n_val], dtype=torch.float32)
y_val   = torch.tensor(y_all[n_train:n_train+n_val], dtype=torch.float32)
X_test  = torch.tensor(X_all[n_train+n_val:],   dtype=torch.float32)
y_test  = torch.tensor(y_all[n_train+n_val:],   dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_train, y_train),
                          batch_size=256, shuffle=True)

print(f"Ventanas totales: {n_total}")
print(f"Entrenamiento: {len(X_train)}   Validacion: {len(X_val)}   Test: {len(X_test)}")
print(f"Forma de X_train: {tuple(X_train.shape)}  (batch, L, 3)")

## 3. Modelo LSTM

La capa `nn.LSTM` de PyTorch implementa la celda con estado de celda y tres compuertas descrita en el capitulo. Tomamos solo la salida del ultimo paso de tiempo (`out[:, -1, :]`) y la proyectamos a los tres componentes del estado siguiente con una capa lineal.

Este es exactamente el modelo del segundo listado del capitulo.

In [ ]:
# ----- Modelo LSTM -----
class LorenzLSTM(nn.Module):
    def __init__(self, input_size=3, hidden_size=64,
                 num_layers=2, output_size=3, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,   # [batch, seq_len, features]
            dropout=dropout
        )
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # x: [batch, L, 3]
        out, _ = self.lstm(x)   # out: [batch, L, hidden_size]
        return self.fc(out[:, -1, :])  # solo el ultimo paso: [batch, 3]

### Bucle de entrenamiento

El entrenamiento usa:

- **Recorte de gradientes** (`clip_grad_norm_`): las RNN son propensas a la explosion del gradiente, especialmente en secuencias largas. Reescala el gradiente cuando su norma supera 1.0.
- **Planificador** `ReduceLROnPlateau`: reduce la tasa de aprendizaje cuando la validacion deja de mejorar.
- **Guardado del mejor modelo** segun la perdida de validacion.

Definimos el bucle como una funcion reutilizable, porque mas adelante entrenaremos tambien una GRU y modelos con distintas longitudes de contexto.

In [ ]:
def entrenar(model, train_loader, X_val, y_val, n_epocas=100,
             lr=1e-3, ruta_guardado=None, verbose=True):
    """Entrena un modelo de secuencia y devuelve el historial de perdidas."""
    model = model.to(dispositivo)
    criterio   = nn.MSELoss()
    optimizador = torch.optim.Adam(model.parameters(), lr=lr)
    planificador = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizador, patience=5, factor=0.5, min_lr=1e-5)

    X_val_d, y_val_d = X_val.to(dispositivo), y_val.to(dispositivo)
    hist_train, hist_val = [], []
    mejor_val_loss = float("inf")

    for epoca in range(1, n_epocas + 1):
        model.train()
        perdida_train = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(dispositivo), yb.to(dispositivo)
            optimizador.zero_grad()
            pred = model(xb)
            perdida = criterio(pred, yb)
            perdida.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizador.step()
            perdida_train += perdida.item() * len(xb)
        perdida_train /= len(train_loader.dataset)

        model.eval()
        with torch.no_grad():
            val_pred  = model(X_val_d)
            perdida_val = criterio(val_pred, y_val_d).item()
        planificador.step(perdida_val)

        hist_train.append(perdida_train)
        hist_val.append(perdida_val)

        if perdida_val < mejor_val_loss:
            mejor_val_loss = perdida_val
            if ruta_guardado is not None:
                torch.save(model.state_dict(), ruta_guardado)

        if verbose and epoca % 10 == 0:
            print(f"Epoca {epoca:3d}: train={perdida_train:.6f}  val={perdida_val:.6f}")

    return {"train": hist_train, "val": hist_val, "mejor_val": mejor_val_loss}

Entrenamos la LSTM. Con GPU son pocos minutos; en CPU tarda mas. Para una ejecucion rapida exploratoria, se puede bajar `N_EPOCAS`.

In [ ]:
N_EPOCAS = 100  # reducir (p. ej. a 30) para una ejecucion rapida

torch.manual_seed(SEMILLA)
modelo_lstm = LorenzLSTM(hidden_size=128, num_layers=2)
hist_lstm = entrenar(modelo_lstm, train_loader, X_val, y_val,
                     n_epocas=N_EPOCAS, ruta_guardado="mejor_lstm_lorenz.pt")
print(f"\nMejor val_loss (LSTM): {hist_lstm['mejor_val']:.6f}")

## 4. Modelo GRU y comparacion con LSTM

La caja de descripcion del capitulo anuncia el entrenamiento de **LSTM y GRU**. La GRU (Gated Recurrent Unit) fusiona el estado de celda y el oculto, y combina las compuertas de olvido y entrada en una unica compuerta de actualizacion. Tiene aproximadamente un 25 % menos de parametros que la LSTM, pero un rendimiento comparable.

En PyTorch basta cambiar `nn.LSTM` por `nn.GRU`: la interfaz es identica. El estado de la GRU es un unico tensor (no la tupla `(h, c)` de la LSTM), pero como aqui solo usamos la salida del ultimo paso, el codigo de `forward` no cambia.

In [ ]:
class LorenzGRU(nn.Module):
    def __init__(self, input_size=3, hidden_size=64,
                 num_layers=2, output_size=3, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.gru(x)            # out: [batch, L, hidden_size]
        return self.fc(out[:, -1, :])   # solo el ultimo paso

torch.manual_seed(SEMILLA)
modelo_gru = LorenzGRU(hidden_size=128, num_layers=2)
hist_gru = entrenar(modelo_gru, train_loader, X_val, y_val,
                    n_epocas=N_EPOCAS, ruta_guardado="mejor_gru_lorenz.pt")
print(f"\nMejor val_loss (GRU): {hist_gru['mejor_val']:.6f}")

In [ ]:
def contar_parametros(m):
    return sum(p.numel() for p in m.parameters())

print(f"Parametros LSTM: {contar_parametros(modelo_lstm):,}")
print(f"Parametros GRU : {contar_parametros(modelo_gru):,}")
print(f"Reduccion GRU vs LSTM: "
      f"{100*(1 - contar_parametros(modelo_gru)/contar_parametros(modelo_lstm)):.1f} %")

### Curvas de aprendizaje

Comparamos la evolucion de la perdida de validacion de ambos modelos.

In [ ]:
plt.figure(figsize=(8, 4))
plt.semilogy(hist_lstm["val"], label="LSTM (validacion)")
plt.semilogy(hist_gru["val"],  label="GRU (validacion)")
plt.xlabel("Epoca")
plt.ylabel("MSE de validacion (escala log)")
plt.title("LSTM frente a GRU: curvas de aprendizaje")
plt.legend()
plt.tight_layout()
plt.show()

### Error de prediccion a un paso en test

Cargamos el mejor modelo guardado de cada arquitectura y medimos el MSE de prediccion a un paso sobre el conjunto de test.

In [ ]:
def mse_test_un_paso(model, ruta, X_test, y_test):
    model.load_state_dict(torch.load(ruta, map_location=dispositivo))
    model.to(dispositivo).eval()
    with torch.no_grad():
        pred = model(X_test.to(dispositivo)).cpu().numpy()
    return ((pred - y_test.numpy())**2).mean()

mse_lstm = mse_test_un_paso(modelo_lstm, "mejor_lstm_lorenz.pt", X_test, y_test)
mse_gru  = mse_test_un_paso(modelo_gru,  "mejor_gru_lorenz.pt",  X_test, y_test)

print(f"MSE test (1 paso) - LSTM: {mse_lstm:.6f}")
print(f"MSE test (1 paso) - GRU : {mse_gru:.6f}")

## 5. Prediccion autorregresiva y horizonte de predictibilidad

Una vez entrenado, el modelo puede usarse en modo **autorregresivo**: la prediccion del paso $t+1$ se anade a la ventana de contexto para predecir el paso $t+2$, y asi sucesivamente, generando trayectorias artificiales de longitud arbitraria.

El sistema de Lorenz es caotico con exponente de Lyapunov $\lambda_1 \approx 0{,}9$ en las unidades de tiempo adimensionales del sistema: los errores se amplifican en un factor $e$ cada $\sim 1{,}1$ unidades de tiempo simulado. Con $\Delta t = 0{,}01$, eso corresponde a $\sim 110$ pasos. El horizonte de predictibilidad de la red (el numero de pasos hasta que el MSE supera un umbral) puede compararse con este limite teorico.

> Tercer listado del capitulo (*Prediccion autorregresiva y horizonte de predictibilidad*).

In [ ]:
def predecir_autorregresivo(model, ventana_inicial, n_predicciones):
    """Genera una trayectoria autorregresiva realimentando cada prediccion."""
    model.eval()
    ventana   = ventana_inicial.clone().to(dispositivo)  # [1, L, 3]
    pred_traj = []
    with torch.no_grad():
        for _ in range(n_predicciones):
            siguiente = model(ventana)          # [1, 3]
            pred_traj.append(siguiente.cpu().numpy())
            # Desplazar ventana: quitar el primer paso, anadir la prediccion
            ventana = torch.cat([ventana[:, 1:, :],
                                 siguiente.unsqueeze(1)], dim=1)
    return np.array(pred_traj).squeeze()        # [n_predicciones, 3]

# Cargar el mejor modelo LSTM y predecir
modelo_lstm.load_state_dict(torch.load("mejor_lstm_lorenz.pt",
                                       map_location=dispositivo))
n_predicciones = 500
pred_traj = predecir_autorregresivo(modelo_lstm, X_test[0:1], n_predicciones)

# Trayectoria real correspondiente (normalizada, para comparar)
real_traj = y_all[n_train+n_val : n_train+n_val+n_predicciones]

# Error cuadratico por paso en unidades normalizadas
mse_por_paso = ((pred_traj - real_traj)**2).mean(axis=1)

# Desnormalizar solo para representar las trayectorias
pred_fis = pred_traj * std + media
real_fis = real_traj * std + media

### Visualizacion del horizonte

A la izquierda, las trayectorias real y predicha (componente $x$) coinciden durante los primeros pasos y luego divergen. A la derecha, el MSE crece de forma aproximadamente exponencial (recta en escala logaritmica), consistente con la dinamica caotica.

In [ ]:
umbral = 1.0
if (mse_por_paso > umbral).any():
    horizonte = int(np.argmax(mse_por_paso > umbral))
else:
    horizonte = len(mse_por_paso)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Componente x: real vs predicha
ax1.plot(real_fis[:, 0], label="real",    lw=1.5)
ax1.plot(pred_fis[:, 0], label="predicha", lw=1.5, linestyle="--")
ax1.axvline(horizonte, color="gray", linestyle=":",
            label=f"horizonte ~ {horizonte} pasos")
ax1.set_xlabel("Pasos de prediccion autorregresiva")
ax1.set_ylabel("x (unidades fisicas)")
ax1.set_title("Trayectoria real frente a predicha")
ax1.legend()

# MSE por paso
ax2.semilogy(mse_por_paso)
ax2.axhline(y=umbral, color="red", linestyle="--",
            label="Umbral de predictibilidad")
ax2.set_xlabel("Pasos de prediccion autorregresiva")
ax2.set_ylabel("MSE (normalizado, escala log)")
ax2.set_title("Horizonte de predictibilidad del sistema de Lorenz")
ax2.legend()

plt.tight_layout()
plt.show()

print(f"Horizonte de predictibilidad (MSE > {umbral}): {horizonte} pasos")
print(f"Limite teorico aproximado (Lyapunov): ~110 pasos")

## 6. Horizonte de predictibilidad en funcion de la longitud del contexto

Esta es la tercera tarea anunciada en la caja: **estudiar el horizonte de predictibilidad en funcion de la longitud del contexto** $L$. Corresponde tambien al ejercicio 10.2 del capitulo.

Entrenamos una LSTM para cada $L \in \{10, 20, 40, 80, 160\}$ y medimos en cada caso el horizonte de predictibilidad autorregresivo. La hipotesis fisica es que, por encima de cierto $L$, aumentar el contexto **deja de mejorar** la prediccion: el limite no es la memoria del modelo, sino el caos intrinseco del sistema (el exponente de Lyapunov).

Usamos modelos algo mas pequenos y menos epocas para que el barrido sea rapido. El patron cualitativo (crecimiento y saturacion) se aprecia igualmente.

In [ ]:
def construir_loaders(L, traj_norm):
    """Construye loaders y tensores con split temporal para un contexto dado."""
    X_all_L, y_all_L = crear_ventanas(traj_norm, L)
    n_tot = len(X_all_L)
    n_tr  = int(0.7 * n_tot)
    n_v   = int(0.15 * n_tot)
    Xtr = torch.tensor(X_all_L[:n_tr],          dtype=torch.float32)
    ytr = torch.tensor(y_all_L[:n_tr],          dtype=torch.float32)
    Xv  = torch.tensor(X_all_L[n_tr:n_tr+n_v],  dtype=torch.float32)
    yv  = torch.tensor(y_all_L[n_tr:n_tr+n_v],  dtype=torch.float32)
    Xte = torch.tensor(X_all_L[n_tr+n_v:],      dtype=torch.float32)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=256, shuffle=True)
    info = {"X_all": X_all_L, "y_all": y_all_L,
            "n_tr": n_tr, "n_v": n_v, "X_test": Xte, "X_val": Xv, "y_val": yv}
    return loader, info

def horizonte_para_L(L, n_epocas=40, n_pred=500, umbral=1.0):
    torch.manual_seed(SEMILLA)
    loader, info = construir_loaders(L, traj_norm)
    modelo = LorenzLSTM(hidden_size=64, num_layers=2)
    entrenar(modelo, loader, info["X_val"], info["y_val"],
             n_epocas=n_epocas, ruta_guardado=None, verbose=False)
    pred = predecir_autorregresivo(modelo, info["X_test"][0:1], n_pred)
    inicio = info["n_tr"] + info["n_v"]
    real = info["y_all"][inicio : inicio + n_pred]
    mse  = ((pred - real)**2).mean(axis=1)
    if (mse > umbral).any():
        return int(np.argmax(mse > umbral))
    return len(mse)

contextos = [10, 20, 40, 80, 160]
horizontes = []
for L_i in contextos:
    h = horizonte_para_L(L_i, n_epocas=40)
    horizontes.append(h)
    print(f"L = {L_i:3d}  ->  horizonte = {h} pasos")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(contextos, horizontes, marker="o")
plt.axhline(y=110, color="red", linestyle="--",
            label="Limite teorico (Lyapunov ~110 pasos)")
plt.xlabel("Longitud del contexto L (pasos)")
plt.ylabel("Horizonte de predictibilidad (pasos)")
plt.title("Horizonte de predictibilidad frente a longitud del contexto")
plt.legend()
plt.tight_layout()
plt.show()

**Interpretacion.** El horizonte crece al aumentar $L$ mientras el contexto aporta informacion util, pero se **satura** una vez que el contexto basta para reconstruir el estado dinamico local. Mas alla de ese punto, el factor limitante no es el modelo sino la sensibilidad exponencial a las condiciones iniciales: la prediccion no puede superar el horizonte impuesto por el exponente de Lyapunov, del orden de 100 pasos para $\Delta t = 0{,}01$. Este es el mensaje fisico central del capitulo: en un sistema caotico, hay un limite duro a la predictibilidad que ninguna arquitectura puede sortear.

## 7. Resumen

En este notebook hemos:

1. **Generado** una trayectoria del atractor de Lorenz integrando sus ecuaciones con Runge-Kutta de cuarto orden, y visualizado su estructura de mariposa.
2. **Preparado** los datos con ventanas deslizantes, split temporal estricto y normalizacion con la estadistica del entrenamiento.
3. **Entrenado y comparado** una LSTM y una GRU: misma interfaz en PyTorch, la GRU con menos parametros y rendimiento comparable.
4. **Estudiado el horizonte de predictibilidad** mediante prediccion autorregresiva, comprobando que crece con la longitud del contexto $L$ hasta saturarse en el limite impuesto por el caos (exponente de Lyapunov).

A partir de aqui, los ejercicios del capitulo extienden el analisis: comparacion con un bosque aleatorio sobre la ventana aplanada (ej. 10.3), arquitecturas seq2seq para prediccion multihorizonte (ej. 10.5) y aplicaciones a series temporales reales (ej. 10.4 y 10.6).